In [13]:
import csv

# Define input files and replicate numbers
files_info = [
    ("/20250405_GeneID_LessTissueV2_2Median-Unique.tsv", 2),
    ("/20250405_GeneID_LessTissueV2_Sampling5-Unique.tsv", 5),
    ("/20250405_GeneID_LessTissueV2_Sampling10-Unique.tsv", 10),
]

for input_file, max_rep in files_info:
    output_file = input_file.replace(".tsv", "_ReDeconv.txt")
    metadata_file = input_file.replace(".tsv", "_Metadata.txt")

    with open(input_file, "r") as fin:
        reader = csv.reader(fin, delimiter='\t')
        header = next(reader)

        new_header = ["GeneID"]
        tissue_counts = {}
        metadata = []

        for tissue in header[1:]:
            tissue_counts[tissue] = tissue_counts.get(tissue, 0) + 1
            replicate_number = tissue_counts[tissue]
            if replicate_number > max_rep:
                continue  # Skip extra columns if they exist
            new_name = f"{tissue}_{replicate_number}"
            new_header.append(new_name)
            metadata.append((new_name, tissue))

        # Write renamed header + data
        with open(output_file, "w", newline='') as fout:
            writer = csv.writer(fout, delimiter='\t')
            writer.writerow(new_header)
            for row in reader:
                # Keep only required number of columns
                writer.writerow([row[0]] + row[1:len(new_header)])

        # Write metadata
        with open(metadata_file, "w", newline='') as fmeta:
            writer = csv.writer(fmeta, delimiter='\t')
            writer.writerow(["Tissue_ID", "Tissue_type"])
            writer.writerows(metadata)

    print(f"Processed '{input_file}' → '{output_file}', metadata saved as '{metadata_file}'")


Processed 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_2Median-Unique.tsv' → 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_2Median-Unique_ReDeconv.txt', metadata saved as 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_2Median-Unique_Metadata.txt'
Processed 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling5-Unique.tsv' → 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling5-Unique_ReDeconv.txt', metadata saved as 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling5-Unique_Metadata.txt'
Processed 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling10-Unique.tsv' → 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling10-Unique_ReDeconv.txt', metadata saved as 'ReDeconv/demo_deconvolution/input/20250405_GeneID_LessTissueV2_Sampling10-Unique_Metadata.txt'


In [14]:
import pandas as pd
import re

# === Step 1: Load genes from both files ===
genes1 = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_2Median-Unique_ReDeconv.txt",
    sep='\t', usecols=[0]
)["GeneID"]

genes2 = pd.read_csv(
    "20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]

set1 = set(genes1)
set2 = set(genes2)
unique_to_file1 = set1 - set2

print(f"Number of genes in first file NOT shared with second file: {len(unique_to_file1)}")

# === Step 2: Load reference (GeneID + GeneName) ===
ref_path = "Raw-Published/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]

# === Step 3: Load GTF and extract GeneID & GeneName ===
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv("gencode.v46.annotation.gtf", sep="\t", comment="#", header=None)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
genes_gtf = gtf[gtf['feature'] == 'gene'].copy()

genes_gtf['GeneID'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf['GeneID_clean'] = genes_gtf['GeneID'].str.split('.').str[0]
genes_gtf['GeneName'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_name"))

# === Step 4: Map unmatched ReDeconv genes to GeneIDs via reference ===
ref_map = dict(zip(reference['GeneName'], reference['GeneID_clean']))
unmatched_to_geneid = {g: ref_map[g] for g in unique_to_file1 if g in ref_map}

# === Step 5: Map those GeneIDs to official GeneNames in GTF ===
gtf_map = dict(zip(genes_gtf['GeneID_clean'], genes_gtf['GeneName']))
unmatched_to_gtf_name = {g: gtf_map[unmatched_to_geneid[g]] for g in unmatched_to_geneid if unmatched_to_geneid[g] in gtf_map}

print(f"Number of unmatched genes that could be remapped using GTF GeneNames: {len(unmatched_to_gtf_name)}")

# === Step 6: Replace in ReDeconv ===
ReDeconv = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_2Median-Unique_ReDeconv.txt",
    sep='\t'
)

ReDeconv['GeneID_original'] = ReDeconv['GeneID']
ReDeconv['GeneID'] = ReDeconv['GeneID'].apply(lambda x: unmatched_to_gtf_name[x] if x in unmatched_to_gtf_name else x)

# === Inspect ===
print(ReDeconv[['GeneID_original', 'GeneID']].head())
ReDeconv = ReDeconv.drop(columns=['GeneID_original'])
ReDeconv.to_csv("/20250405_GeneID_LessTissueV2_2Median-Unique_ReDeconv_Renamed.txt", sep='\t', index=False)


Number of genes in first file NOT shared with second file: 20100


/tmp/ipykernel_3684095/2031211666.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Number of unmatched genes that could be remapped using GTF GeneNames: 19358
  GeneID_original       GeneID
0         DDX11L1      DDX11L1
1          WASH7P       WASH7P
2       MIR6859-1    MIR6859-1
3     MIR1302-2HG  MIR1302-2HG
4         FAM138A      FAM138A


In [15]:
import pandas as pd
import re

# === Step 1: Load genes from both files ===
genes1 = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_Sampling5-Unique_ReDeconv.txt",
    sep='\t', usecols=[0]
)["GeneID"]

genes2 = pd.read_csv(
    "20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]

set1 = set(genes1)
set2 = set(genes2)
unique_to_file1 = set1 - set2

print(f"Number of genes in first file NOT shared with second file: {len(unique_to_file1)}")

# === Step 2: Load reference (GeneID + GeneName) ===
ref_path = "Raw-Published/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]

# === Step 3: Load GTF and extract GeneID & GeneName ===
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv("gencode.v46.annotation.gtf", sep="\t", comment="#", header=None)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
genes_gtf = gtf[gtf['feature'] == 'gene'].copy()

genes_gtf['GeneID'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf['GeneID_clean'] = genes_gtf['GeneID'].str.split('.').str[0]
genes_gtf['GeneName'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_name"))

# === Step 4: Map unmatched ReDeconv genes to GeneIDs via reference ===
ref_map = dict(zip(reference['GeneName'], reference['GeneID_clean']))
unmatched_to_geneid = {g: ref_map[g] for g in unique_to_file1 if g in ref_map}

# === Step 5: Map those GeneIDs to official GeneNames in GTF ===
gtf_map = dict(zip(genes_gtf['GeneID_clean'], genes_gtf['GeneName']))
unmatched_to_gtf_name = {g: gtf_map[unmatched_to_geneid[g]] for g in unmatched_to_geneid if unmatched_to_geneid[g] in gtf_map}

print(f"Number of unmatched genes that could be remapped using GTF GeneNames: {len(unmatched_to_gtf_name)}")

# === Step 6: Replace in ReDeconv ===
ReDeconv = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_Sampling5-Unique_ReDeconv.txt",
    sep='\t'
)

ReDeconv['GeneID_original'] = ReDeconv['GeneID']
ReDeconv['GeneID'] = ReDeconv['GeneID'].apply(lambda x: unmatched_to_gtf_name[x] if x in unmatched_to_gtf_name else x)

# === Inspect ===
print(ReDeconv[['GeneID_original', 'GeneID']].head())
ReDeconv = ReDeconv.drop(columns=['GeneID_original'])
ReDeconv.to_csv("/20250405_GeneID_LessTissueV2_Sampling5-Unique_ReDeconv_Renamed.txt", sep='\t', index=False)


Number of genes in first file NOT shared with second file: 20100


/tmp/ipykernel_3684095/3997625430.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Number of unmatched genes that could be remapped using GTF GeneNames: 19358
  GeneID_original       GeneID
0         DDX11L1      DDX11L1
1          WASH7P       WASH7P
2       MIR6859-1    MIR6859-1
3     MIR1302-2HG  MIR1302-2HG
4         FAM138A      FAM138A


In [16]:
import pandas as pd
import re

# === Step 1: Load genes from both files ===
genes1 = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_Sampling10-Unique_ReDeconv.txt",
    sep='\t', usecols=[0]
)["GeneID"]

genes2 = pd.read_csv(
    "20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]

set1 = set(genes1)
set2 = set(genes2)
unique_to_file1 = set1 - set2

print(f"Number of genes in first file NOT shared with second file: {len(unique_to_file1)}")

# === Step 2: Load reference (GeneID + GeneName) ===
ref_path = "Raw-Published/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]

# === Step 3: Load GTF and extract GeneID & GeneName ===
def extract_gtf_info(attr, key):
    match = re.search(f'{key} "([^"]+)"', attr)
    return match.group(1) if match else None

gtf = pd.read_csv("gencode.v46.annotation.gtf", sep="\t", comment="#", header=None)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']
genes_gtf = gtf[gtf['feature'] == 'gene'].copy()

genes_gtf['GeneID'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_id"))
genes_gtf['GeneID_clean'] = genes_gtf['GeneID'].str.split('.').str[0]
genes_gtf['GeneName'] = genes_gtf['attribute'].apply(lambda x: extract_gtf_info(x, "gene_name"))

# === Step 4: Map unmatched ReDeconv genes to GeneIDs via reference ===
ref_map = dict(zip(reference['GeneName'], reference['GeneID_clean']))
unmatched_to_geneid = {g: ref_map[g] for g in unique_to_file1 if g in ref_map}

# === Step 5: Map those GeneIDs to official GeneNames in GTF ===
gtf_map = dict(zip(genes_gtf['GeneID_clean'], genes_gtf['GeneName']))
unmatched_to_gtf_name = {g: gtf_map[unmatched_to_geneid[g]] for g in unmatched_to_geneid if unmatched_to_geneid[g] in gtf_map}

print(f"Number of unmatched genes that could be remapped using GTF GeneNames: {len(unmatched_to_gtf_name)}")

# === Step 6: Replace in ReDeconv ===
ReDeconv = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_Sampling10-Unique_ReDeconv.txt",
    sep='\t'
)

ReDeconv['GeneID_original'] = ReDeconv['GeneID']
ReDeconv['GeneID'] = ReDeconv['GeneID'].apply(lambda x: unmatched_to_gtf_name[x] if x in unmatched_to_gtf_name else x)

# === Inspect ===
print(ReDeconv[['GeneID_original', 'GeneID']].head())
ReDeconv = ReDeconv.drop(columns=['GeneID_original'])
ReDeconv.to_csv("/20250405_GeneID_LessTissueV2_Sampling10-Unique_ReDeconv_Renamed.txt", sep='\t', index=False)


Number of genes in first file NOT shared with second file: 20100


/tmp/ipykernel_3684095/780744723.py:23: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Number of unmatched genes that could be remapped using GTF GeneNames: 19358
  GeneID_original       GeneID
0         DDX11L1      DDX11L1
1          WASH7P       WASH7P
2       MIR6859-1    MIR6859-1
3     MIR1302-2HG  MIR1302-2HG
4         FAM138A      FAM138A


In [20]:
import pandas as pd
import re

# === Load genes from both files ===
genes1 = pd.read_csv(
    "/20250405_GeneID_LessTissueV2_Sampling10-Unique_ReDeconv_Renamed.txt",
    sep='\t', usecols=[0]
)["GeneID"]

genes2 = pd.read_csv(
    "20250616_All-Tissues-NoDup_Random_Simulated_v2_Counts.txt",
    sep='\t', usecols=[0]
)["Geneid"]

set1 = set(genes1)
set2 = set(genes2)

print(f"Number of genes in first file: {len(set1)}")
print(f"Number of shared genes: {len(set1.intersection(set2))}")

# === Get genes unique to file 1 ===
unique_to_file1 = set1 - set2
print(f"Number of genes in first file NOT shared with second file: {len(unique_to_file1)}")

# === Load reference dataset with GeneID_clean and GeneName ===
ref_path = "Raw-Published/20250405_Both_LessTissueV2_2Median-Unique.tsv"
reference = pd.read_csv(ref_path, sep='\t', index_col=False)
reference.columns.values[1] = "GeneName"
reference = reference.iloc[:, :2]
reference['GeneID_clean'] = reference['GeneID'].astype(str).str.split('.').str[0]

# === Make mapping dictionary GeneName -> GeneID_clean ===
gene_name_to_id = dict(zip(reference['GeneName'], reference['GeneID_clean']))

# === Find GeneID_clean for unmatched genes using the dictionary ===
unmatched_genes_ids = [gene_name_to_id[gn] for gn in unique_to_file1 if gn in gene_name_to_id]

# === Load GTF and filter ===
gtf_path = "gencode.v46.annotation.gtf"
gtf = pd.read_csv(gtf_path, sep="\t", comment="#", header=None, low_memory=False)
gtf.columns = ['seqname', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute']

# Extract gene_id from GTF and remove version
def extract_gene_id_clean(attr):
    match = re.search('gene_id "([^"]+)"', attr)
    if match:
        return match.group(1).split('.')[0]  # Remove version after '.'
    return None

gtf['GeneID_clean'] = gtf['attribute'].apply(extract_gene_id_clean)
genes_gtf = gtf[gtf['feature'] == 'gene']

# Filter GTF for unmatched GeneIDs (without version)
gtf_matched = genes_gtf[genes_gtf['GeneID_clean'].isin(unmatched_genes_ids)]
print(f"Number of unmatched GeneID_clean found in GTF: {len(gtf_matched)}")


Number of genes in first file: 54551
Number of shared genes: 53809
Number of genes in first file NOT shared with second file: 742


/tmp/ipykernel_3684095/2881295437.py:27: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  reference = pd.read_csv(ref_path, sep='\t', index_col=False)


Number of unmatched GeneID_clean found in GTF: 0
